<a href="https://colab.research.google.com/github/HarithaGottumukkala/DATA266-1598_hw3/blob/main/Homework3_1598.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA 266 — Homework 3
## Prompt Engineering and Self-Attention

### Personal Parameters

| Parameter | Value |
|---|---:|
| SID4 | 1598 |
| SEED | 1598 |
| SLICE | 598 |
| HP_ID | 2 |
| CLS_A | 8 |
| CLS_B | 5 |

I report all six personal parameters in the first cell, as required
by Homework 1 Step 0. I calculate them from my student ID in the
next code cell.

For Homework 3, I use SEED = 1598 for reproducibility. The assignment
does not specify a use for the other derived parameters.

## 0. Assignment Overview

In Part A, I will compare six prompt-engineering techniques using
a pretrained language model from Hugging Face running inside Colab.
I will call the model through LangChain.

Each technique will have two separate prompt examples, giving
12 experiments. I will display the actual responses and compare
their correctness, explanations, and structure.

In Part B, I will implement single-head scaled dot-product
self-attention using basic PyTorch operations and the exact text
provided by the professor.

I will train two models:
1. An unmasked attention model.
2. A causal attention model that cannot attend to future positions.

Both models will learn through next-token prediction. After training,
I will visualize their attention weights and check the causal mask
numerically.

I will base my findings on the actual outputs and follow the
standing requirements for reproducibility, run logs, metrics,
model checkpoints, AI-use documentation, and repository submission.

In [1]:
SID4 = 1598

SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print("Personal assignment parameters")
print("SID4:", SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)

Personal assignment parameters
SID4: 1598
SEED: 1598
SLICE: 598
HP_ID: 2
CLS_A: 8
CLS_B: 5


## 0.1 Reproducibility and Environment

I use seed 1598 for Python, NumPy, and PyTorch, following the
Homework 1 standing instructions.

A random seed controls the sequence of random numbers used by
the code. This helps repeat an experiment under the same software
and hardware conditions.

I will reset the seed before creating each attention model so
their initial parameters are comparable. Both models will use
the same training settings, with causal masking as the intended
difference.

Homework 3 does not request a multi-seed experiment, so I will
use the assigned seed for both models.

The next code cell will import the required libraries, set the
seeds, and print the software versions and device information.

I will use the CPU for the small attention experiment. I will
check the pretrained language model's device and response time
separately when setting up the prompt experiments.

In [2]:
import sys
import random
import math
import re
from datetime import datetime, timezone

import numpy as np
import torch
from torch import nn
import matplotlib
import matplotlib.pyplot as plt


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)

# The small attention models will run on the CPU.
device = torch.device("cpu")

run_started_at = datetime.now(timezone.utc).isoformat()

print("Setup time (UTC):", run_started_at)
print("Python version:", sys.version.split()[0])
print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)
print("Matplotlib version:", matplotlib.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Attention model device:", device)
print("Main random seed:", SEED)

Setup time (UTC): 2026-09-12T16:20:21.905979+00:00
Python version: 3.13.15
NumPy version: 2.1.3
PyTorch version: 2.11.0+cpu
Matplotlib version: 3.10.0
CUDA available: False
Attention model device: cpu
Main random seed: 1598


## 1. Prompt Engineering

In this section, I will study how changing a prompt affects a
language model's response.

I will use a pretrained model downloaded from Hugging Face and
run it inside Colab. I will call it programmatically through
LangChain and display its responses in the notebook.

### Experiment Design

I will use the same two base tasks across all six techniques:
one math problem and one logical-reasoning problem.

Using the same tasks will help me compare the effect of the
prompting technique. I will also keep the model and generation
settings consistent.

| Technique | How I will write the prompt |
|---|---|
| Zero-Shot | Ask the question directly without examples. |
| Few-Shot | Provide example questions and answers before the task. |
| Chain-of-Thought | Provide a worked example with concise solution steps before the task. |
| Zero-Shot CoT | Request concise solution steps without providing examples. |
| Meta-Prompting | Ask the model to identify a suitable strategy and apply it. |
| Tree of Thoughts | Ask for candidate approaches, brief evaluations, and a selected solution. |

Each technique will have two separate code cells, giving
12 prompt experiments in total. The Tree of Thoughts examples
will use a simple prompt-based exploration of alternatives.

### What I Will Compare

After running the prompts, I will compare:

- Whether the answers are correct.
- Whether the model follows the requested format.
- How much explanation it provides.
- Whether examples or alternative approaches help.

I will write observations from the actual responses. A technique
does not have to improve the answer for the result to be useful.

### 1.1 Package Setup

The next code cell will install the Hugging Face and LangChain
packages needed to load and call the pretrained model.

This pretrained model is used for the prompt experiments.
For the attention experiment, I will implement the attention
calculations myself using basic PyTorch operations.

In [3]:
%pip install -q langchain-huggingface transformers accelerate

from importlib.metadata import version

from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print("Package versions:")
print("langchain-huggingface:", version("langchain-huggingface"))
print("transformers:", version("transformers"))
print("accelerate:", version("accelerate"))

print("\nAll required imports succeeded.")

Package versions:
langchain-huggingface: 1.2.2
transformers: 5.16.1
accelerate: 1.14.0

All required imports succeeded.


### 1.2 Model Choice and Generation Settings

I will use Qwen2.5-1.5B-Instruct from Hugging Face for all
12 prompt experiments.

This is a pretrained, instruction-tuned language model with
approximately 1.54 billion parameters. Instruction tuning prepares
a model to respond to user instructions.

The model will run inside my Colab runtime. Downloading its files
requires internet access.

I will use the following generation settings:

- do_sample = False: select the highest-scoring next token instead
  of randomly sampling a token.
- max_new_tokens = 384: allow up to 384 generated tokens per response.

A token can be a word, part of a word, or punctuation, so this
limit is not the same as 384 words.

Keeping the model and settings consistent helps me compare the
prompting techniques. These settings reduce randomness, but they
do not guarantee correct answers.

Before starting the graded experiments, I will run a short test
to check that the model responds and measure its response time.

CPU generation may be slow. I will check the actual runtime
before deciding whether this setup is suitable for my live demo.

In [4]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

set_seed(SEED)

print("Loading tokenizer...")
llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading the language model on the CPU...")
llm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32,
)

llm_model = llm_model.to("cpu")
llm_model.eval()

print("\nModel loaded successfully.")
print("Model name:", MODEL_ID)
print("Model device:", next(llm_model.parameters()).device)
print("Model data type:", next(llm_model.parameters()).dtype)
print("Model revision:", getattr(llm_model.config, "_commit_hash", "unavailable"))

Loading tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Model loaded successfully.
Model name: Qwen/Qwen2.5-1.5B-Instruct
Model device: cpu
Model data type: torch.float32
Model revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306


### 1.3 LangChain Setup and a Short Model Test

Next, I will connect the loaded model to LangChain using
HuggingFacePipeline. I will reuse one LangChain model object
for all 12 prompt experiments.

The input will follow this path:

Prompt text → model's chat format → tokenizer → language model → response

The chat format marks which text is the user's message and where
the model should begin its answer. I will use the tokenizer's
built-in chat template to apply the format expected by Qwen.

I will turn off random sampling and allow up to 384 new tokens
per response, as described in the previous section.

Before starting the experiments, I will send one short test prompt
and measure how long it takes to receive the answer.

This setup test is separate from the 12 graded prompt experiments.
It checks that the complete calling process works. It does not
establish the model's accuracy or speed on longer problems.

I will inspect the actual response before writing any observations.

In [5]:
from time import perf_counter

# Create the text-generation pipeline with fixed settings.
text_generator = pipeline(
    task="text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    max_new_tokens=384,
    do_sample=False,
    temperature=None,
    top_p=None,
    top_k=None,
    return_full_text=False,
    pad_token_id=llm_tokenizer.eos_token_id,
)

# Reuse this object for all 12 graded experiments.
llm = HuggingFacePipeline(pipeline=text_generator)

test_prompt = "What is 2 + 2? Reply with only the number."

# Format the message using the model's expected chat structure.
test_messages = [
    {"role": "user", "content": test_prompt}
]

formatted_test_prompt = llm_tokenizer.apply_chat_template(
    test_messages,
    tokenize=False,
    add_generation_prompt=True,
)

start_time = perf_counter()
test_response = llm.invoke(formatted_test_prompt)
test_seconds = perf_counter() - start_time

print("Test prompt:", test_prompt)
print("Model response:", test_response.strip())
print(f"Response time: {test_seconds:.2f} seconds")

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'top_p', 'top_k', 'do_sample', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). S

Test prompt: What is 2 + 2? Reply with only the number.
Model response: 4
Response time: 6.79 seconds


### 1.4 The Two Questions

I will ask the same two questions using all six prompting techniques.
This will help me see what changes when I change the prompt.

#### Question A — Splitting a Pizza Bill

My friend and I order two pizzas that cost $12 each.
We get a 20% discount on the pizzas. After the discount,
the restaurant adds a $4 delivery fee. There are no other charges.

If we split the final bill equally, how much does each person pay?

The two pizzas cost $24. The discount is $4.80, so the pizzas
cost $19.20 after the discount. Adding delivery makes the total
$23.20. Splitting that between two people gives $11.60 each.

**Expected answer: $11.60 per person.**

#### Question B — Using a Study Room

A student can use a study room if they have both a student ID
and a room booking. Otherwise, they cannot use it.

- Anu has a student ID but no booking.
- Ravi has a booking but no student ID.
- Meera has both a student ID and a booking.

Who is allowed to use the study room?

**Expected answer: Only Meera.**

Meera has both things needed to use the room. Anu and Ravi
each have only one.

#### What I Will Check

I will check whether the model gets these answers right.
If it gives an explanation, I will check that too.

I will also compare whether its answers are clear, whether it
follows my instructions, and whether the extra explanation helps.

In [6]:
task_a = """My friend and I order two pizzas that cost $12 each.
We get a 20% discount on the pizzas. After the discount,
the restaurant adds a $4 delivery fee. There are no other charges.

If we split the final bill equally, how much does each person pay?"""

task_b = """A student can use a study room if they have both a
student ID and a room booking. Otherwise, they cannot use it.

- Anu has a student ID but no booking.
- Ravi has a booking but no student ID.
- Meera has both a student ID and a booking.

Who is allowed to use the study room?"""

# These answers are for checking the model's responses.
reference_answers = {
    "A": "$11.60 per person",
    "B": "Only Meera",
}

prompt_results = []

print("QUESTION A")
print(task_a)

print("\nQUESTION B")
print(task_b)

print("\nExpected answer A:", reference_answers["A"])
print("Expected answer B:", reference_answers["B"])

QUESTION A
My friend and I order two pizzas that cost $12 each.
We get a 20% discount on the pizzas. After the discount,
the restaurant adds a $4 delivery fee. There are no other charges.

If we split the final bill equally, how much does each person pay?

QUESTION B
A student can use a study room if they have both a
student ID and a room booking. Otherwise, they cannot use it.

- Anu has a student ID but no booking.
- Ravi has a booking but no student ID.
- Meera has both a student ID and a booking.

Who is allowed to use the study room?

Expected answer A: $11.60 per person
Expected answer B: Only Meera


### 1.5 Zero-Shot Prompting

Zero-shot prompting means asking the model a question without
giving it any examples first.

I will send each question directly to the model. I will not
provide a worked example or ask for a particular solution method.

This gives me a starting point for comparison with the other
prompting techniques.

#### Example 1 — Splitting a Pizza Bill

First, I will ask the pizza question.

I will check whether the model applies the discount only to
the pizzas, adds the delivery fee, and splits the total correctly.

The expected answer is $11.60 per person. I will keep this
reference answer out of the prompt.

In [7]:
# Ask the question directly, without examples.
prompt_zero_shot_b = task_b

messages = [
    {"role": "user", "content": prompt_zero_shot_b}
]

formatted_prompt = llm_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

print("PROMPT")
print(prompt_zero_shot_b)

start_time = perf_counter()
response_zero_shot_b = llm.invoke(formatted_prompt).strip()
elapsed_seconds = perf_counter() - start_time

print("\nMODEL RESPONSE")
print(response_zero_shot_b)
print(f"\nResponse time: {elapsed_seconds:.2f} seconds")

# Save the actual output for our later comparison.
prompt_results.append({
    "technique": "Zero-Shot",
    "task": "B",
    "prompt": prompt_zero_shot_b,
    "response": response_zero_shot_b,
    "time_seconds": elapsed_seconds,
    "word_count": len(response_zero_shot_b.split()),
})

[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PROMPT
A student can use a study room if they have both a
student ID and a room booking. Otherwise, they cannot use it.

- Anu has a student ID but no booking.
- Ravi has a booking but no student ID.
- Meera has both a student ID and a booking.

Who is allowed to use the study room?

MODEL RESPONSE
To determine who is allowed to use the study room, we need to check each individual's possession of both a student ID and a room booking:

1. **Anu**:
   - Has a student ID.
   - Does not have a room booking.
   - Therefore, Anu does not meet the requirement for using the study room because they do not have a room booking.

2. **Ravi**:
   - Does not have a student ID.
   - Has a room booking.
   - Therefore, Ravi does not meet the requirement for using the study room because they do not have a student ID.

3. **Meera**:
   - Has both a student ID and a room booking.
   - Therefore, Meera meets the requirement for using the study room.

Based on this analysis, **Meera** is the only one who i